# QA Generation from documents: trials

following https://huggingface.co/learn/cookbook/rag_evaluation 

In [ ]:
import os
import textwrap
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib.pyplot as plt
import numpy as np
from langchain_ollama import ChatOllama
from tqdm import tqdm

from Wallis.evaluation.QA_creation import *
from Wallis.RAGs.RAGv3 import VectorBase, build_graph

In [ ]:
%load_ext autoreload

In [ ]:
def print_wrap(text, width=80):
    wrapped = textwrap.fill(text, width=width)
    print(wrapped)

In [ ]:
# to avoid useless prints
import logging

logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

In [ ]:
vectorebase = VectorBase(
    directory="../transformed_documents",
    chunk_size=5000,
    chunk_overlap=300,
)

In [ ]:
# distribution of length of documents
all_lengths = []
for doc_path in os.listdir("../transformed_documents"):
    with open(f"../transformed_documents/{doc_path}") as f:
        doc = f.read()
    all_lengths.append(len(doc.split()))

In [ ]:
plt.hist(all_lengths, bins=100)
plt.title("Distribution of document lengths (words)")
plt.xlabel("Document length (words)")
plt.ylabel("Frequency")
plt.xscale("log")
median = np.median(all_lengths)
third_quartile = np.percentile(all_lengths, 75)
plt.axvline(third_quartile, color="green", label=f"Third quartile: {third_quartile}")
plt.axvline(median, color="red", label=f"Median: {median}")
plt.legend()
plt.show()

take chunks

In [ ]:
chunks = vectorebase.chunks

Prompts tested

In [ ]:
QA_generation_prompt = """
I will give you a context, your task is to write a short and concise True or False question about this context. It must be factual and answerable with True or False and should not be over 8 words.
Write the question in french as the context is in french but the answer in english (True or False).
Output:::
Factoid question: (your factoid question)
Answee: (your answer to the factoid question, must be True, or False)

Now here is the context.

Context: {context}\n
Output:::"""

In [ ]:
QA_generation_prompt = """
Your task is to write a factoid question and an answer given a context.
Your factoid question should be about a specific, concise piece of factual information from the context.
Your factoid question should be formulated in the same style as questions users could ask in a search engine, maximum 9 words.
This means that your factoid question MUST NOT mention something like "according to the passage" or "context".
Write the question and the answer in french.
The question should contain enough part of the context to be answerable without the context. 
For example, you can start your question with "Dans le cadre de..." if you think it is necessary to give context.
"Quels sont les frais de photocopie ?" is a bad question, as it is not answerable without the context.
"Quels sont les frais de photocopie à la bibliothèque de Sion ?" is a good question, as it is answerable without the context.
Provide your answer as follows, please do not add any spaces character and absolutely respect this format, do not include anything else in your answer:
Output:::
Question: (your factoid question with enough context to be answerable without the context)
Réponse: (your answer to the factoid question)

Now here is the context.

Context: {context}\n
Output:::"""

In [ ]:
llm = ChatOllama(model="llama3.1:8b", num_predict=200)

## QA generation

In [ ]:
def generate_questions(chunk, llm, verbose=False):
    """Generate question fron a chunk. A chunk is a FAISS Document instance, llm is a ChatOllama instance"""
    context = chunk.page_content
    # print_wrap(context[:500], width=50)
    whole_prompt = QA_generation_prompt.format(context=context)
    # print(whole_prompt)
    generated_questions = llm.invoke(whole_prompt)
    try:
        if verbose:
            print(generated_questions.content)
        if "Question:" in generated_questions.content:
            splitted = generated_questions.content.split("Question:")[1]
        else:
            splitted = generated_questions.content.split("Question :")[1]
        if "Réponse:" in splitted:
            [question, answer] = splitted.split("Réponse:")
        else:
            [question, answer] = splitted.split("Réponse :")
    except:
        print("Error")
        print(generated_questions.content)
        question = "Error"
        answer = "Error"

    question = question.strip()
    answer = answer.strip()
    dico_res = {
        "title": chunk.metadata["title"],
        "question": question,
        "answer": answer,
        "context": context,
    }
    return dico_res

check results

In [ ]:
rqndom_chunk = np.random.choice(chunks)
print(rqndom_chunk.metadata["title"])
generate_questions(rqndom_chunk, llm)

In [ ]:
generated_questions = []
sample_chunks = np.random.choice(chunks, 100)
for chunk in tqdm(sample_chunks, total=len(sample_chunks)):
    res = generate_questions(chunk, llm)
    generated_questions.append(res)

try parralelization

In [ ]:
def process_chunk(chunk):
    return generate_questions(chunk, llm)


results = []
with ThreadPoolExecutor(max_workers=6) as executor:
    future_to_sample = {
        executor.submit(process_chunk, chunk): chunk for chunk in sample_chunks
    }

    for future in tqdm(as_completed(future_to_sample), total=len(future_to_sample)):
        results.append(future.result())

## QA assessement (setup critic agents)

In [ ]:
question_groundedness_critique_prompt = """
You will be given a context and a question, both in french.
Your task is to provide a 'total rating' scoring how well one can answer the given question unambiguously with the given context.
Give your answer on a scale of 1 to 5, where 1 means that the question is not answerable at all given the context, and 5 means that the question is clearly and unambiguously answerable with the context.

Provide your answer as follows:

Answer:::
Evaluation: (your short rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here are the question and context.

Question: {question}\n
Context: {context}\n
Answer::: """

question_relevance_critique_prompt = """
You will be given a question in french.
Your task is to provide a 'total rating' representing how useful this question can be to someone looking for information or procedure about laws of the canton du Valais in Switzerland.
Give your answer on a scale of 1 to 5, where 1 means that the question is not useful at all, and 5 means that the question is extremely useful.

Provide your answer as follows:

Answer:::
Evaluation: (your short rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here is the question.

Question: {question}\n
Answer::: """

In [ ]:
for question in tqdm(generated_questions, total=len(generated_questions)):
    prompt_groudness = question_groundedness_critique_prompt.format(
        question=question["question"], context=question["context"]
    )
    prompt_relevance = question_relevance_critique_prompt.format(
        question=question["question"]
    )
    groundness = llm.invoke(prompt_groudness)
    relevance = llm.invoke(prompt_relevance)
    # print_wrap(question["question"])
    # print_wrap(question["answer"])
    # print_wrap(question["context"][:500])
    print("-" * 25)
    try:
        if "Total rating:" in groundness.content:
            groundness = groundness.content.split("Total rating: ")[1]
        else:
            groundness = groundness.content.split("Total rating : ")[1]
        # groundness=groundness.split("\n")[0]
        groundness = float(groundness)
        print(f"Groundness: {groundness}")
    except:
        print("Error")
        print(groundness)
        groundness = 0
    print("-" * 25)
    try:
        if "Total rating:" in relevance.content:
            relevance = relevance.content.split("Total rating: ")[1]
            # relevance=relevance.split("\n")[0]
        else:
            relevance = relevance.content.split("Total rating : ")[1]
            relevance = float(relevance)
        print(f"Relevance: {relevance}")
    except:
        print("Error")
        print(relevance)
        print("-" * 70)

In [ ]:
llm_groundness = ChatOllama(model="llama3.1:8b", num_predict=200)
llm_relevance = ChatOllama(model="llama3.1:8b", num_predict=200)


def grade_generated_questions(generated_questions):
    for question in tqdm(generated_questions, total=len(generated_questions)):
        prompt_groundness = question_groundedness_critique_prompt.format(
            question=question["question"], context=question["context"]
        )
        prompt_relevance = question_relevance_critique_prompt.format(
            question=question["question"]
        )

        def get_groundness():
            return llm_groundness.invoke(prompt_groundness)

        def get_relevance():
            return llm_relevance.invoke(prompt_relevance)

        # Run in parallel
        with ThreadPoolExecutor() as executor:
            future_groundness = executor.submit(get_groundness)
            future_relevance = executor.submit(get_relevance)

            # Get results
            groundness = future_groundness.result()
            relevance = future_relevance.result()
        try:
            if "Total rating:" in groundness.content:
                groundness = groundness.content.split("Total rating: ")[1]
            else:
                groundness = groundness.content.split("Total rating : ")[1]
            groundness = float(groundness)
            # print(f"Groundness: {groundness}")
        except:
            print("Error")
            print(groundness)
            groundness = 0
        try:
            if "Total rating:" in relevance.content:
                relevance = relevance.content.split("Total rating: ")[1]
            else:
                relevance = relevance.content.split("Total rating : ")[1]
            relevance = float(relevance)
            # print(f"Relevance: {relevance}")
        except:
            print("Error")
            print(relevance)
            relevance = 0
            print("-" * 70)
        question["groundness"] = groundness
        question["relevance"] = relevance
    return generated_questions

In [ ]:
graded_questions = grade_generated_questions(generated_questions)

In [ ]:
# save list of graded questions
import json

with open("intermediate_results/graded_questions.jsonl", "w", encoding="utf-8") as f:
    json.dump(graded_questions, f, ensure_ascii=False)

In [ ]:
len(graded_questions)

In [ ]:
# load list of graded questions
import json

with open("intermediate_results/graded_questions.jsonl", "r", encoding="utf-8") as f:
    graded_questions = json.load(f)

In [ ]:
with open("intermediate_results/graded_questions.jsonl", "w", encoding="utf-8") as f:
    json.dump(graded_questions, f, ensure_ascii=False)

function to filter questions (questions already graded)

In [ ]:
def filter_questions(graded_questions, min_groundness=3, min_relevance=3):
    filtered_questions = []
    for question in graded_questions:
        if (
            question["groundness"] >= min_groundness
            and question["relevance"] >= min_relevance
        ):
            filtered_questions.append(question)
    return filtered_questions

In [ ]:
filtered_questions = filter_questions(
    graded_questions, min_groundness=3, min_relevance=3
)

In [ ]:
len(filtered_questions)

## Evaluate RAG

In [ ]:
generic_answer = "okok on est là pour ça"

try proetheus eval

In [ ]:
from prometheus_eval import PrometheusEval
from prometheus_eval.prompts import ABSOLUTE_PROMPT, SCORE_RUBRIC_TEMPLATE
from prometheus_eval.vllm import VLLM

In [ ]:
import torch

torch.cuda.empty_cache()

In [ ]:
model = VLLM(model="prometheus-eval/prometheus-7b-v2.0")

In [ ]:
judge = PrometheusEval(model=model, absolute_grade_template=ABSOLUTE_PROMPT)

In [ ]:
rubric_data = {
    "criteria": "Is the response correct, accurate, and factual based on the reference answer?",
    "score1_description": "The response is completely incorrect, inaccurate, and/or not factual.",
    "score2_description": "The response is mostly incorrect, inaccurate, and/or not factual.",
    "score3_description": "The response is somewhat correct, accurate, and/or factual.",
    "score4_description": "The response is mostly correct, accurate, and factual.",
    "score5_description": "The response is completely correct, accurate, and factual.",
}
score_rubric = SCORE_RUBRIC_TEMPLATE.format(**rubric_data)

In [ ]:
res = graded_questions[np.random.choice(len(graded_questions))]
print(res["title"])
print(res["question"])
print(res["answer"])
instruction = res["question"]
answer = res["answer"]

In [ ]:
reference_answer = "12 janvier 1991"
feedback, score = judge.single_absolute_grade(
    instruction=instruction,
    response=answer,
    rubric=score_rubric,
    reference_answer=reference_answer,
)

print("Feedback:", feedback)
print("Score:", score)

#### Evaluate with llama (much faster)

first prompt is without the context (but eval is bad)

In [ ]:
JUDGE_PROMPT = """
You will be given a user question, a true answer and a generated answer.
Your task is to provide a 'total rating' scoring how well the generated answer answers the user question compared to the truth.
Give your answer as a float on a scale of 0 to 10, where 0 means that the generated answer is not helpful at all and entirely false, and 10 means that the answer completely addresses the question and matches the true answer.

Provide your feedback as follows:

Feedback:::
Evaluation: (your short rationale for the rating, as a text)
Total rating: (your rating, as a float between 0 and 10)

Now here are the question , the true answer and the generated answer.

Question: {question}
True answer: {true_answer}
Generated answer: {generated_answer}

Feedback:::
Total rating: """

In [ ]:
llm_judge = ChatOllama(model="llama3.1:8b", num_predict=200, temperature=0)

In [ ]:
sample = graded_questions[np.random.choice(len(graded_questions))]
question = sample["question"]
print(question)
true_answer = sample["answer"]
print(true_answer)

In [ ]:
generated_answer = "Le service juridique des affaires économiques et le service de la protection des travailleurs"
prompt = JUDGE_PROMPT.format(
    question=question, true_answer=true_answer, generated_answer=generated_answer
)

feedback = llm_judge.invoke(prompt)
print(feedback.content)
rating = feedback.content.split("Total rating: ")[1]
rating = float(rating)
print(f"Parsed rating: {rating}")

In [ ]:
generic_answer = "22"

In [ ]:
def evaluate_generated_answers(generated_answers):
    llm_judge = ChatOllama(model="llama3.1:8b", num_predict=200, temperature=0)
    for sample in tqdm(generated_answers, total=len(generated_answers)):
        question = sample["question"]
        generated_answer = sample["answer"]
        true_answer = generic_answer
        prompt = JUDGE_PROMPT.format(
            question=question,
            true_answer=true_answer,
            generated_answer=generated_answer,
        )
        feedback = llm_judge.invoke(prompt)
        rating = feedback.content.split("Total rating: ")[1]
        rating = float(rating)
        sample["rating"] = rating
    return generated_answers

In [ ]:
generated_answers_graded = evaluate_generated_answers(graded_questions)

In [ ]:
all_ratings = [sample["rating"] for sample in generated_answers_graded]
plt.hist(all_ratings, bins=100)
plt.title("Distribution of ratings")
plt.xlabel("Rating")
plt.ylabel("Frequency")
plt.show()

parralelization

In [ ]:
llm_judge = ChatOllama(model="llama3.1:8b", num_predict=200, temperature=0)


def evaluate_sample(sample):
    """Evaluate a single generated answer."""
    question = sample["question"]
    generated_answer = sample["answer"]
    if "truth" in sample:
        true_answer = sample["truth"]
    else:
        true_answer = generic_answer  # Assuming this is defined elsewhere
    prompt = JUDGE_PROMPT.format(
        question=question, true_answer=true_answer, generated_answer=generated_answer
    )

    feedback = llm_judge.invoke(prompt)
    rating = float(feedback.content.split("Total rating: ")[1])

    sample["rating"] = rating
    return sample


def evaluate_generated_answers_parallel(generated_answers, num_workers=4):
    """Parallel evaluation using ThreadPoolExecutor."""
    results = []

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        future_to_sample = {
            executor.submit(evaluate_sample, sample): sample
            for sample in generated_answers
        }

        for future in tqdm(
            as_completed(future_to_sample),
            total=len(generated_answers),
            desc="Evaluating",
        ):
            results.append(future.result())

    return results


def evaluate_generated_answers_from_lists(questions, true_answers, generated_answers):
    answers_good_format = [
        {"question": question, "answer": answer, "truth": true_answer}
        for question, true_answer, answer in zip(
            questions, true_answers, generated_answers
        )
    ]
    return evaluate_generated_answers_parallel(answers_good_format)

In [ ]:
evaluated_answers = evaluate_generated_answers_parallel(graded_questions, num_workers=6)

#### IMPORT RAG

In [ ]:
chunk_overlap = 300
chunk_size = [500, 1000]
DIRECTORY = "../transformed_documents"
top_k = 5
model_name = "llama3.1:8b"
num_predict_tokens = 200

In [ ]:
vector_base = VectorBase(
    directory=DIRECTORY, chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

In [ ]:
graph = build_graph(
    vector_base, model_name=model_name, num_predict=num_predict_tokens, top_k=top_k
)
config = {"configurable": {"thread_id": "abc123"}}

In [ ]:
questions = [sample["question"] for sample in evaluated_answers]
questions = questions[:100]

get answers fron the RAG

In [ ]:
def get_answers_from_rag(graph, config, questions):
    answers = []
    for question in tqdm(questions, total=len(questions)):
        input_message = question
        input_message = input_message.replace("'", " ").replace("’", " ")
        for step in graph.stream(
            {"messages": [{"role": "user", "content": input_message}]},
            stream_mode="updates",
            config=config,
        ):
            pass
            # print("heeeeeeeeeere")
            # print(step)
            # answers.append(step["messages"][-1].content)
        if "generate" in step:
            # print(step["generate"]["messages"][0].content)
            answers.append(step["generate"]["messages"][0].content)
        else:
            print("Error")
            print(step)
            answers.append("Error")

    return answers

In [ ]:
results = get_answers_from_rag(graph, config, questions)

In [ ]:
true_answers = [sample["answer"] for sample in evaluated_answers]
true_answers = true_answers[:100]

evaluate RAG answers

In [ ]:
evaluations = evaluate_generated_answers_from_lists(questions, true_answers, results)

In [ ]:
plt.hist([sample["rating"] for sample in evaluations], bins=20)
plt.title("Distribution of ratings")
plt.xlabel("Rating")
plt.ylabel("Frequency")
median = np.median([sample["rating"] for sample in evaluations])
plt.axvline(median, color="red", label=f"Median: {median}")
plt.legend()
plt.show()